In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from listUtils import getFlatList
from master import MasterParams, MusicDBPermDir
from base import MusicDBDir, MusicDBData
from sys import prefix
from pandas import Series, DataFrame, concat
from match import MatchListDataNames, MatchListDataRefs
from musicdb import PanDBIO
from lib import rateyourmusic

# Functions

In [ ]:
def getURL(ref):
    retval = ref.replace("-","_").replace("/artist/", "").upper()
    return retval

def getRYMName(artistName):
    retval = artistName.split(" [")[0].strip().upper() if artistName.endswith("]") else artistName.upper()
    return retval

def fixFloat(x):
    try:
        retval = int(x)
    except:
        retval = x
    return retval

def removeKnownRefs(refs):
    if isinstance(refs,list):
        refs = [ref for ref in refs if rymRefs.get(ref.replace("-", "_")) is None]        
        retval = []
        for ref in refs:
            refname = ref.replace("-", "_")
            if masterRefs.get(refname) is not None:
                continue
            masterRefs[refname] = True
            retval.append(ref)
        return retval
    return []

def getUniqueRefs(urlsToGet):
    dupls = {}
    urefs = {}
    for key,row in urlsToGet.iterrows():
        refs = row["Ref"]
        save = []
        for ref in refs:
            if dupls.get(ref) is not None:
                continue
            dupls[ref] = True
            save.append(ref)
        urefs[key] = save
    retval = Series(urefs, name="URefs")
    return retval

def fixArtistRefs(refs):
    df = DataFrame([x.get().values() for x in getFlatList(refs.values)])
    df.index = df[0]
    df.index.name = ""
    df.columns = ["Ref", "ArtistID", "Name", "None"]
    df = df.drop(["Ref",'None'], axis=1)
    df["ArtistID"] = df["ArtistID"].apply(lambda x: x[7:-1] if isinstance(x,str) else x)
    df["N"] = 1
    return df

# Find URLs From Lists To Download

In [ ]:
#############################################################################################################################################################################################
#############################################################################################################################################################################################
mio      = rateyourmusic.MusicDBIO(verbose=False)
refData  = mio.data.getSummaryRefData()
pdbio    = PanDBIO()
mmeDF    = pdbio.getData()
mmeDF["Rank"] = mmeDF["PrimaryRank"].apply(lambda rank: rank[0])
rymID    = mmeDF[mmeDF["RateYourMusic"].notna()]["RateYourMusic"]
unknown  = mmeDF[mmeDF["RateYourMusic"].isna()][["ArtistName", "Rank", "Counts"]]

#############################################################################################################################################################################################
#############################################################################################################################################################################################
minCounts = None
maxCounts = None
minRank   = 30000
maxRank   = 31000

#############################################################################################################################################################################################
#############################################################################################################################################################################################
print("="*200)
artistsToMatch = unknown
print("Found {0: >6} Unmatched Master Artsts".format(artistsToMatch.shape[0]))
if isinstance(minCounts,int):
    artistsToMatch = artistsToMatch[(artistsToMatch["Counts"] > minCounts)]
    print("Found {0: >6} Unmatched Master Artsts With Counts > {1}".format(artistsToMatch.shape[0], minCounts))
if isinstance(maxCounts,int):
    artistsToMatch = artistsToMatch[(artistsToMatch["Counts"] <= maxCounts)]
    print("Found {0: >6} Unmatched Master Artsts With Counts <= {1}".format(artistsToMatch.shape[0], maxCounts))
if isinstance(minRank,int):
    artistsToMatch = artistsToMatch[(artistsToMatch["Rank"] > minRank)]
    print("Found {0: >6} Unmatched Master Artsts With Rank > {1}".format(artistsToMatch.shape[0], minRank))
if isinstance(maxRank,int):
    artistsToMatch = artistsToMatch[(artistsToMatch["Rank"] <= maxRank)]
    print("Found {0: >6} Unmatched Master Artsts With Rank <= {1}".format(artistsToMatch.shape[0], maxRank))
print("="*200)


In [ ]:
mdbpd = MusicDBPermDir()
permDBDir = mdbpd.getDBPermPath("RateYourMusic")
listData = io.get(permDBDir.join("rym/charts.p"))
if False:
    listData["Ref"] = listData.index
    listData.index = listData["Ref"].map(getURL)
    listData.index.name = ""
    listData = listData[~listData.index.duplicated()]

#############################################################################################################################################################################################
print("="*200)
minLists  = 1
maxLists  = 20000000

#############################################################################################################################################################################################
print("="*200)
ignores  = ['/artist/various-artists', '/artist/cap_1', '/artist/frank-capp', '/artist/sean_p', '/artist/killer-4']
print("Found {0} RateYourMusic Downloaded Artists".format(len(refData)))
print("Found {0} RateYourMusic Matched Artists".format(len(rymID)))
print("Found {0} RateYourMusic List Artists".format(listData.shape[0]))
print("="*200)

#############################################################################################################################################################################################
print("="*200)
listDataToUse = listData
print(f"Found {listDataToUse.shape[0]} List Artists")
listDataToUse = listDataToUse[~listDataToUse["ArtistID"].isin(rymID)]
print(f"Found {listDataToUse.shape[0]} List Artists (Not Matched)")
listDataToUse = listDataToUse[~listDataToUse["ArtistID"].isin(refData.index)]
print(f"Found {listDataToUse.shape[0]} List Artists (Not Matched, Not Downloaded)")
listDataToUse = listDataToUse[~listDataToUse.index.isin(ignores)]
print(f"Found {listDataToUse.shape[0]} List Artists (Not Matched, Not Downloaded, Not Ignored)")
localIgnores =  ['/artist/kaleidon']
localIgnores += ['/artist/daryll-ann', '/artist/the_alexander_brothers', '/artist/seth', '/artist/s_type', '/artist/the-rainmakers', '/artist/ken_thorne', '/artist/unruly_child', '/artist/megafaun', '/artist/lian_ross', '/artist/joel-ford', '/artist/the-servants', '/artist/joe_grind', '/artist/leo-anibaldi', '/artist/suppuration_f2', '/artist/when-in-rome', '/artist/mabel_mercer', '/artist/bocafloja', '/artist/porter', '/artist/epsilon', '/artist/kim_boyce', '/artist/ich___ich', '/artist/cahit_berkay', '/artist/larry_finnegan', '/artist/billy_bob_thornton', '/artist/mortal_love', '/artist/that-kid', '/artist/spax', '/artist/the-devil-dogs', '/artist/the_ones', '/artist/pitboss-2000', '/artist/frederick-knight', '/artist/frank-wilson', '/artist/defiance', '/artist/joss_whedon', '/artist/the_bots', '/artist/bochum_welt', '/artist/j_d__blackfoot', '/artist/crazy_ken_band', '/artist/monumentum', '/artist/carmen_rizzo', '/artist/young-marco-1', '/artist/a_shoreline_dream', '/artist/delta_v', '/artist/andrew-dice-clay', '/artist/infinity-frequencies', '/artist/lyriel', '/artist/siobhan_donaghy', '/artist/el_gran_silencio', '/artist/plants_and_animals', '/artist/pirana', '/artist/kraken', '/artist/end_of_fashion', '/artist/susan_aglukark', '/artist/wendy-matthews', '/artist/alexa-5', '/artist/foje', '/artist/charles_de_goal', '/artist/the_mynabirds', '/artist/dj_matrix', '/artist/curt-boettcher', '/artist/foday_musa_suso', '/artist/novika', '/artist/the_spinanes', '/artist/yonderboi', '/artist/soon_e_mc', '/artist/the_lambrettas', '/artist/hate_dept_', '/artist/j__j__jackson', '/artist/branko-1', '/artist/the_sports', '/artist/celluloide', '/artist/streetwalkers', '/artist/steve_james', '/artist/bengalfuel', '/artist/fistula', '/artist/harry_beckett', '/artist/mazhar_alanson', '/artist/fukpig', '/artist/horrorpops', '/artist/alpay', '/artist/michael_keaton', '/artist/michael_katon', '/artist/helmut-zacharias', '/artist/adhesive', '/artist/ghostek', '/artist/meg', '/artist/the_bathers', '/artist/jakalope', '/artist/disastrous-murmur', '/artist/lugubrum', '/artist/alex-brown', '/artist/show_ya', '/artist/coma_cinema', '/artist/del-mccoury', '/artist/andy_irvine', '/artist/craig_richards', '/artist/slick_idiot', '/artist/the_tears', '/artist/johnny_fuller', '/artist/sierra_maestra']
localIgnores += ['/artist/d_lo', '/artist/will_brandes', '/artist/lushlife', '/artist/pride_of_lions', '/artist/concrete-sox', '/artist/anette-olzon', '/artist/paul_kossoff', '/artist/warumpi_band', '/artist/goldroom', '/artist/le_kid', '/artist/los_incas', '/artist/straight_line_stitch', '/artist/bergthron', '/artist/steve-walsh', '/artist/roland_w_', '/artist/brenda_and_the_tabulations', '/artist/the_virgins_f1', '/artist/david_naughton', '/artist/tranquility-bass', '/artist/enochian-crescent', '/artist/victor_ark', '/artist/vulcain', '/artist/the_eternal', '/artist/the_lazy_cowgirls', '/artist/lee_noble', '/artist/t_raumschmiere', '/artist/alyson_williams', '/artist/al-tall', '/artist/kari_rueslatten', '/artist/kashmere_stage_band', '/artist/optimus_rhyme', '/artist/los-crudos', '/artist/easy_going', '/artist/david_parsons', '/artist/christine_martin', '/artist/guiomar-novaes', '/artist/zombie_girl', '/artist/koushik', '/artist/willem-breuker', '/artist/avenue_d', '/artist/riistetyt', '/artist/gregg_kowalsky', '/artist/skogen_f1', '/artist/jacknife-lee', '/artist/jackie-lee', '/artist/jim_pembroke', '/artist/nervochaos', '/artist/max_corbacho', '/artist/colin_bass', '/artist/lisa_maffia', '/artist/nathan_abshire', '/artist/tacere', '/artist/credit_to_the_nation', '/artist/tittsworth', '/artist/houwitser', '/artist/louis-van-dijk', '/artist/mark_johnson', '/artist/the_49ers', '/artist/zooey-deschanel', '/artist/abdominal-1', '/artist/the_kingstonians', '/artist/timoria', '/artist/rich-medina', '/artist/isacaarum', '/artist/hangar', '/artist/gehenna-1', '/artist/lou-johnson-2', '/artist/john-campbell', '/artist/chuck_wicks', '/artist/loveholics', '/artist/amanda', '/artist/jesse_johnson', '/artist/choking-victim', '/artist/spitfire-6', '/artist/hardcell', '/artist/judy-clay', '/artist/judy_clay', '/artist/parental-advisory', '/artist/akurat', '/artist/azrael_f7', '/artist/converter', '/artist/the_critters', '/artist/camoflauge-monk', '/artist/maniac', '/artist/general_echo', '/artist/yyrkoon', '/artist/veikko_lavi', '/artist/jack_greene', '/artist/defiled', '/artist/eddie_warner', '/artist/paul-sirrell', '/artist/bebu_silvetti', '/artist/dilermando_reis', '/artist/emjay', '/artist/mindrot', '/artist/ripcord', '/artist/the_nutmegs', '/artist/phil-ranelin', '/artist/harry_stewart', '/artist/maximum-joy']
#listDataToUse = listDataToUse[~listDataToUse.index.isin(localIgnores)]
print(f"Found {listDataToUse.shape[0]} List Artists (Not Matched, Not Downloaded, Not Ignored)")
print("="*200)


#############################################################################################################################################################################################
print("="*200)
print("Found {0: >6} Total Chart Artists".format(listDataToUse.shape[0]))
listDataToGet = listDataToUse[(listDataToUse["N"] < maxLists) & (listDataToUse["N"] >= minLists)]
print("Found {0: >6} Total Chart Artists With {1} <= N < {2}".format(listDataToGet.shape[0], minLists, maxLists))
print("="*200)

In [ ]:
from match import poolMatchNames
retval = poolMatchNames(baseNames=artistsToMatch["ArtistName"].map(getRYMName).drop_duplicates(), compNames=listDataToGet["Name"].map(getRYMName), nCores=3, progress=True, cutoff=0.95)
results = retval[retval.map(len) > 0]
matchedListArtists = results.apply(lambda x: list(x.index))
matchedListArtists.name = "Ref"
urlsToGet = artistsToMatch.loc[results.index].join(matchedListArtists)
print(f"Found {urlsToGet.shape[0]} Matched Artist Results") 

urefs = getUniqueRefs(urlsToGet)
urlsToGet = urlsToGet.join(urefs)
nAllRefs = urlsToGet["Ref"].map(len).sum()
nAllURefs = urlsToGet["URefs"].map(len).sum()
print(f"Found {nAllURefs}/{nAllRefs} Refs To Download")
io.save(idata=urlsToGet, ifile="urlsToGet25000.p")

# Find URLs From Relationships To Download

In [ ]:
#############################################################################################################################################################################################
#############################################################################################################################################################################################
mio      = rateyourmusic.MusicDBIO(verbose=False)
refData  = mio.data.getSummaryRefData()
pdbio    = PanDBIO()
mmeDF    = pdbio.getData()
mmeDF["Rank"] = mmeDF["PrimaryRank"].apply(lambda rank: rank[0])
rymID    = mmeDF[mmeDF["RateYourMusic"].notna()]["RateYourMusic"]
unknown  = mmeDF[mmeDF["RateYourMusic"].isna()][["ArtistName", "Rank", "Counts"]]

#############################################################################################################################################################################################
#############################################################################################################################################################################################
minCounts = None
maxCounts = None
minRank   = 30000
maxRank   = 31000

#############################################################################################################################################################################################
#############################################################################################################################################################################################
print("="*200)
artistsToMatch = unknown
print("Found {0: >6} Unmatched Master Artsts".format(artistsToMatch.shape[0]))
if isinstance(minCounts,int):
    artistsToMatch = artistsToMatch[(artistsToMatch["Counts"] > minCounts)]
    print("Found {0: >6} Unmatched Master Artsts With Counts > {1}".format(artistsToMatch.shape[0], minCounts))
if isinstance(maxCounts,int):
    artistsToMatch = artistsToMatch[(artistsToMatch["Counts"] <= maxCounts)]
    print("Found {0: >6} Unmatched Master Artsts With Counts <= {1}".format(artistsToMatch.shape[0], maxCounts))
if isinstance(minRank,int):
    artistsToMatch = artistsToMatch[(artistsToMatch["Rank"] > minRank)]
    print("Found {0: >6} Unmatched Master Artsts With Rank > {1}".format(artistsToMatch.shape[0], minRank))
if isinstance(maxRank,int):
    artistsToMatch = artistsToMatch[(artistsToMatch["Rank"] <= maxRank)]
    print("Found {0: >6} Unmatched Master Artsts With Rank <= {1}".format(artistsToMatch.shape[0], maxRank))
print("="*200)

In [ ]:
from lib.rateyourmusic import Relationships
rts = Relationships()
mem = rts.getMem()
mof = rts.getMof()
asa = rts.getAsa()
rar = rts.getRar()
refdf = concat([fixArtistRefs(mem["Refs"]), fixArtistRefs(mof["Refs"]), fixArtistRefs(rar["Refs"]), fixArtistRefs(asa["Refs"])]).drop_duplicates()
print("Found {0: >6} Total Relationship Artists".format(refdf.shape[0]))

In [ ]:
from match import poolMatchNames
retval = poolMatchNames(baseNames=artistsToMatch["ArtistName"].map(getRYMName).drop_duplicates(), compNames=refdf["Name"].map(getRYMName), nCores=3, progress=True, cutoff=0.95)
results = retval[retval.map(len) > 0]
matchedRelationArtists = results.apply(lambda x: list(x.index))
matchedRelationArtists.name = "Ref"
urlsToGet = artistsToMatch.loc[results.index].join(matchedRelationArtists)
print(f"Found {urlsToGet.shape[0]} Matched Artist Results") 

urefs = getUniqueRefs(urlsToGet)
urlsToGet = urlsToGet.join(urefs)
nAllRefs = urlsToGet["Ref"].map(len).sum()
nAllURefs = urlsToGet["URefs"].map(len).sum()
print(f"Found {nAllURefs}/{nAllRefs} Refs To Download")
io.save(idata=urlsToGet, ifile="urlsToGet25000.p")

# Download Found URLs

In [ ]:
from ioutils import FileIO
from gate import IOStore
from pandas import Series
from numpy import ceil
io        = FileIO()
ios       = IOStore()
mdbio     = ios.get("RateYourMusic")
refData   = mdbio.data.getSummaryRefData()
rymRefs   = refData.apply(lambda x: x.replace("https://rateyourmusic.com", "").replace("-", "_") if isinstance(x,str) else None)
rymRefs   = {v: k for k,v in rymRefs[rymRefs.notna()].to_dict().items()}

In [ ]:
from pandas import concat
#urlsToGet = concat([io.get("urlsToGet2.1.p"), io.get("urlsToGet1.1.p")])
urlsToGet = io.get("urlsToGet25000.p")
urlsToGet = urlsToGet.sort_values(by="Rank", ascending=True)
print(f"# Found {len(rymRefs)} Known RateYourMusic Refs")
print(f"# Found {urlsToGet.shape[0]} Artists URLs To Get")
masterRefs = {}
urlsToGet["ToGet"] = urlsToGet["URefs"].apply(lambda x: removeKnownRefs(x))
done = urlsToGet[urlsToGet["ToGet"].map(len) == 0]
print(f"# Found {done.shape[0]} Artists With Known URLs")
refsToGet = urlsToGet[(urlsToGet["ToGet"].map(len) > 0)] # & urlsToGet["Rank"] > 132036]
print(f"# Found {refsToGet.shape[0]} Artists URLs To Download")

# Found 2773 Artists URLs To Get
# Found 77 Artists With Known URLs
# Found 2696 Artists URLs To Download

In [ ]:
head = 12
hset = 12
N    = refsToGet.shape[0]
nT   = int(ceil(N/hset))
for i,(_,row) in enumerate(refsToGet[((head-1)*hset):((head)*hset)].iterrows()):
    refs   = row["URefs"]
    name   = row["ArtistName"]
    rank   = fixFloat(row["Rank"])
    #if rank < 357559: continue
    counts = fixFloat(row["Counts"])
    n      = (head-1)*hset+i+1
    for ref in refs:
        url    = "https://rateyourmusic.com{0}".format(ref)
        print(f"{head: >3} / {nT: <3} | {n: >5} / {N: <4} |  {name: <40}{rank: <10}{counts: <4} | {url}")

# Check Unknown (For New List)

In [ ]:
from musicdb import PanDBIO
pdbio    = PanDBIO()
pdbio.addMetrics()
pdbio.setIndex()

In [ ]:
from musicdb import PanDBIO
pdbio    = PanDBIO()
mmeDF    = pdbio.getData()
mmeDF["Rank"] = mmeDF["PrimaryRank"].apply(lambda rank: rank[0])
rymID    = mmeDF[mmeDF["RateYourMusic"].notna()]["RateYourMusic"]
unknown  = mmeDF[mmeDF["RateYourMusic"].isna()][["ArtistName", "Rank", "PrimaryRank", "Counts"]]

In [ ]:
unknown[(unknown["Rank"] > 18750) & (unknown["Rank"] <= 25000)]

# Parse

In [ ]:
from lib import rateyourmusic
rateyourmusic.moveLocalFiles()
rateyourmusic.removeLocalFiles()

from utils import PoolIO
pio = PoolIO("RateYourMusic")
pio.parse()
pio.meta()
pio.sum()
pio.search()

from lib.rateyourmusic import Relationships
rts = Relationships()
rts.compute()